# 03 Comparable Bond Selection

This notebook ranks peer bonds for each target using simple, explainable similarity rules and uses those peers to infer a fair spread.

The point of this step is not to claim that the scoring system is perfect. It is to show a clear and defendable process for identifying relevant peers, weighting them, and turning that peer set into a fair-value anchor.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.comps import infer_fair_spreads, select_top_comps

In [3]:
analytics_path = PROJECT_ROOT / "outputs" / "bond_analytics.csv"
universe_path = PROJECT_ROOT / "data" / "raw" / "bond_universe.csv"

analytics = pd.read_csv(analytics_path, parse_dates=["evaluation_date", "maturity_date"])
universe = pd.read_csv(universe_path, parse_dates=["evaluation_date", "maturity_date"])
bonds = analytics.merge(
    universe,
    on=["bond_id", "issuer", "evaluation_date", "maturity_date", "coupon_rate", "clean_price"],
    how="left",
)
bonds[["bond_id", "issuer", "rating", "sector", "spread_to_curve"]]

,bond_id,issuer,rating,sector,spread_to_curve
0,AMZN_2031_425,Amazon.com Inc,A,Consumer Cyclical,0.004108
1,AMZN_2036_488,Amazon.com Inc,A,Consumer Cyclical,0.006153
2,F_2030_400,Ford Motor Credit Company LLC,BBB,Consumer Cyclical,0.016347
3,F_2033_713,Ford Motor Credit Company LLC,BBB,Consumer Cyclical,0.020095
4,META_2030_420,Meta Platforms Inc,AA,Communications,0.004290
5,META_2035_488,Meta Platforms Inc,AA,Communications,0.007494
6,PFE_2033_475,Pfizer Investment Enterprises Pte,A,Consumer Non-Cyclical,0.006182
7,PFE_2053_530,Pfizer Investment Enterprises Pte,A,Consumer Non-Cyclical,0.010392


In [4]:
top_comps = select_top_comps(bonds, max_comps=3)
top_comps

,target_bond_id,comp_bond_id,target_issuer,comp_issuer,issuer_match,rating_match,seniority_match,sector_match,currency_match,maturity_gap_years,coupon_gap_pct,spread_gap_bps,comp_score,comp_weight
0,AMZN_2031_425,AMZN_2036_488,Amazon.com Inc,Amazon.com Inc,1,1,1,1,1,5.002053,0.625,20.450410,4.572345,4.572345
1,AMZN_2031_425,META_2030_420,Amazon.com Inc,Meta Platforms Inc,0,0,1,0,1,0.323066,0.050,1.821923,2.161889,2.161889
2,AMZN_2031_425,PFE_2033_475,Amazon.com Inc,Pfizer Investment Enterprises Pte,0,1,1,0,1,2.184805,0.500,20.742298,1.879887,1.879887
3,AMZN_2036_488,AMZN_2031_425,Amazon.com Inc,Amazon.com Inc,1,1,1,1,1,5.002053,0.625,20.450410,4.572345,4.572345
4,AMZN_2036_488,PFE_2033_475,Amazon.com Inc,Pfizer Investment Enterprises Pte,0,1,1,0,1,2.817248,0.125,0.291888,2.174945,2.174945
5,AMZN_2036_488,META_2035_488,Amazon.com Inc,Meta Platforms Inc,0,0,1,0,1,0.325804,0.000,13.410553,1.837040,1.837040
6,F_2030_400,F_2033_713,Ford Motor Credit Company LLC,Ford Motor Credit Company LLC,1,1,1,1,1,2.984257,3.125,37.487542,4.425468,4.425468
7,F_2030_400,AMZN_2031_425,Ford Motor Credit Company LLC,Amazon.com Inc,0,0,1,1,1,0.328542,0.250,122.387542,-0.559460,0.100000
8,F_2030_400,META_2030_420,Ford Motor Credit Company LLC,Meta Platforms Inc,0,0,1,0,1,0.005476,0.200,120.565619,-1.221349,0.100000
9,F_2033_713,F_2030_400,Ford Motor Credit Company LLC,Ford Motor Credit Company LLC,1,1,1,1,1,2.984257,3.125,37.487542,4.425468,4.425468


In [5]:
fair_spreads = infer_fair_spreads(bonds, top_comps)
fair_spreads

,bond_id,fair_spread,comp_count,comp_support_score,comp_ids
0,AMZN_2031_425,0.005692,3,2.871374,"AMZN_2036_488, META_2030_420, PFE_2033_475"
1,AMZN_2036_488,0.005358,3,2.861443,"AMZN_2031_425, PFE_2033_475, META_2035_488"
2,F_2030_400,0.019408,3,0.881553,"F_2033_713, AMZN_2031_425, META_2030_420"
3,F_2033_713,0.015907,3,-0.832518,"F_2030_400, PFE_2033_475, AMZN_2036_488"
4,META_2030_420,0.006341,3,2.004952,"META_2035_488, AMZN_2031_425, PFE_2033_475"
5,META_2035_488,0.004878,3,2.029900,"META_2030_420, AMZN_2036_488, PFE_2033_475"
6,PFE_2033_475,0.005260,3,1.368539,"AMZN_2036_488, AMZN_2031_425, META_2035_488"
7,PFE_2053_530,0.006610,3,-10.452428,"PFE_2033_475, AMZN_2036_488, META_2035_488"


In [6]:
comp_view = top_comps[["target_bond_id", "comp_bond_id", "comp_score", "maturity_gap_years", "spread_gap_bps"]].copy()
comp_view.sort_values(["target_bond_id", "comp_score"], ascending=[True, False])

,target_bond_id,comp_bond_id,comp_score,maturity_gap_years,spread_gap_bps
0,AMZN_2031_425,AMZN_2036_488,4.572345,5.002053,20.450410
1,AMZN_2031_425,META_2030_420,2.161889,0.323066,1.821923
2,AMZN_2031_425,PFE_2033_475,1.879887,2.184805,20.742298
3,AMZN_2036_488,AMZN_2031_425,4.572345,5.002053,20.450410
4,AMZN_2036_488,PFE_2033_475,2.174945,2.817248,0.291888
5,AMZN_2036_488,META_2035_488,1.837040,0.325804,13.410553
6,F_2030_400,F_2033_713,4.425468,2.984257,37.487542
7,F_2030_400,AMZN_2031_425,-0.559460,0.328542,122.387542
8,F_2030_400,META_2030_420,-1.221349,0.005476,120.565619
9,F_2033_713,F_2030_400,4.425468,2.984257,37.487542


In a production setting, comp selection would likely bring in more issuer history, sector conventions, liquidity signals, and trade evidence. For this project, the simpler rule set is useful because every part of the decision can be inspected directly.